# Semantic Job Search — Embeddings + Vector DB
### Pune AI Builders Meetup — Notebook 06

---

> **Where we left off:** The complete agent in NB05 works — but `search_jobs` is naive.
> It matches keywords against a `skills` column. On real job data, that column is usually empty.
>
> **This notebook:** Replace keyword search with semantic search using embeddings + ChromaDB.
> Same tool interface. Dramatically better results. The agent doesn't know anything changed.

---
## Step 1 — The Problem with Keyword Search

Let's load the real job database and run the NB05 keyword search against it.

In [1]:
OPENAI_API_KEY = "sk-proj-****************"  # ← replace with your OpenAI API key

In [2]:
import os, json
import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY))

JOB_DATABASE = pd.read_csv("../data/jobs.csv")
print(f"Loaded {len(JOB_DATABASE)} jobs")
print(f"Columns: {list(JOB_DATABASE.columns)}")
print()

# How many jobs actually have a skills field?
skills_filled = JOB_DATABASE["skills"].fillna("").str.strip()
has_skills = skills_filled != ""
print(f"Jobs with skills field populated : {has_skills.sum()} / {len(JOB_DATABASE)}")
print(f"Jobs with empty skills field     : {(~has_skills).sum()} / {len(JOB_DATABASE)}")
print()

# How many have a job_description?
desc_filled = JOB_DATABASE["job_description"].fillna("").str.strip()
has_desc = desc_filled != ""
print(f"Jobs with job_description        : {has_desc.sum()} / {len(JOB_DATABASE)}")

Loaded 500 jobs
Columns: ['id', 'job_uid', 'job_title', 'company_name', 'location', 'job_type', 'experience_required', 'skills', 'salary', 'job_description', 'apply_link', 'posted_date', 'expiry_date', 'source_website', 'source_name', 'validity_status', 'is_active', 'created_at', 'updated_at', 'last_checked_at']

Jobs with skills field populated : 0 / 500
Jobs with empty skills field     : 500 / 500

Jobs with job_description        : 497 / 500


In [3]:
# ── NB05 keyword search — copied exactly ──────────────────────────────────
def search_jobs_keyword(skills: list, domain: str = None, min_years: int = 0) -> list:
    skill_set = set(s.strip().lower() for s in skills)
    results = []
    for _, row in JOB_DATABASE[JOB_DATABASE["is_active"] == True].iterrows():
        raw_skills = row["skills"]
        if not isinstance(raw_skills, str) or not raw_skills.strip():
            continue  # skip jobs with no skills field
        required = set(s.strip().lower() for s in raw_skills.split(","))
        overlap  = len(skill_set & required) / len(required)
        if overlap >= 0.4 and row["experience_required"] <= min_years:
            results.append({
                "job_title"  : row["job_title"],
                "company"    : row["company_name"],
                "match_score": round(overlap * 100),
            })
    return sorted(results, key=lambda x: x["match_score"], reverse=True)[:5]


# Arjun's profile — backend engineer, 8 years, Java/Go/Kafka
ARJUN_SKILLS = ["Java", "Go", "Kafka", "Redis", "distributed systems"]

results = search_jobs_keyword(skills=ARJUN_SKILLS, min_years=8)

print(f"Keyword search results: {len(results)}")
if results:
    for r in results:
        print(f"  {r['match_score']}%  {r['job_title']} @ {r['company']}")
else:
    print()
    print("  Zero results.")
    print()
    print("  The skills column is empty on real scraped jobs.")
    print("  Our tool is blind — it can't see 499 real opportunities.")

Keyword search results: 0

  Zero results.

  The skills column is empty on real scraped jobs.
  Our tool is blind — it can't see 499 real opportunities.


---
## Step 2 — What Are Embeddings?

An embedding is a vector — a list of numbers — that encodes the **meaning** of text.
Two texts with similar meaning produce vectors that point in similar directions.

We measure similarity with **cosine similarity**: 1.0 = identical, 0.0 = unrelated.

```
"distributed systems"      →  [0.12, -0.45, 0.88, ...]  ─┐ cosine ≈ 0.85 (similar)
"platform engineering"     →  [0.15, -0.41, 0.82, ...]  ─┘

"distributed systems"      →  [0.12, -0.45, 0.88, ...]
"product roadmap planning" →  [0.61,  0.33, 0.02, ...]    cosine ≈ 0.18 (unrelated)
```

Keyword search can't see this — it only knows if strings match exactly.

In [4]:
import numpy as np

# Helper used throughout the notebook
def embed(text: str) -> list:
    """Embed a single string using text-embedding-3-small."""
    resp = client.embeddings.create(model="text-embedding-3-small", input=[text])
    return resp.data[0].embedding

def cosine_similarity(a: list, b: list) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Compare four sentences — see how cosine score reflects meaning
sentences = [
    "distributed systems and platform engineering",
    "backend infrastructure, Kafka, event-driven architecture",
    "machine learning model training and data pipelines",
    "product roadmap planning and stakeholder management",
]

query_vec = embed(sentences[0])

print(f"Query: '{sentences[0]}'")
print()
print(f"{'Sentence':<55} {'Similarity':>10}")
print("-" * 68)
for sent in sentences:
    score = cosine_similarity(query_vec, embed(sent))
    bar   = "█" * int(score * 30)
    print(f"{sent:<55} {score:>6.3f}  {bar}")

Query: 'distributed systems and platform engineering'

Sentence                                                Similarity
--------------------------------------------------------------------
distributed systems and platform engineering             1.000  █████████████████████████████
backend infrastructure, Kafka, event-driven architecture  0.503  ███████████████
machine learning model training and data pipelines       0.324  █████████
product roadmap planning and stakeholder management      0.351  ██████████


---
## Step 3 — Chunking Strategy: LLM-Extracted Chunks

Before embedding, we need to decide **what text** to embed per job.

Real job descriptions are ~6 KB of mixed content:
```
[Company intro boilerplate]    ← ~1 500 chars, zero signal
[Responsibilities]             ← moderate signal  
[Requirements / qualifications]← HIGH signal — directly matches candidate skills
[EEO / legal / agency tail]   ← ~800 chars, zero signal
```

**Why not regex / rule-based splitting?**
Section headers vary wildly across 500 jobs and 20+ companies.
Regex only works on structure. An LLM works on **meaning**.

**Our strategy: pre-compute chunks with an LLM (one-time cost)**
```
jobs.csv  →  GPT-4o-mini (one call per job)  →  jobs_chunks.csv
```
- Extract: role, required skills, experience, domain
- Discard: boilerplate, EEO text, benefits, generic phrases
- Cost: ~$0.14 for all 500 jobs — run once, commit the file, never pay again
- Embedding code reads `jobs_chunks.csv` — clean separation of concerns

The script lives at `scripts/generate_chunks.py`. Run it once before building the index.

In [5]:
from pathlib import Path

CHUNKS_CSV = Path("../data/jobs_chunks.csv")

# ── Show the LLM prompt used to generate chunks ────────────────────────────
CHUNK_PROMPT = """
You are a technical recruiter assistant extracting a clean indexing chunk from a job description.

Extract and include:
  - Role title, seniority, and company domain
  - Required technical skills and tools (languages, frameworks, platforms)  
  - Years of experience required
  - 1-2 lines of key responsibilities (specific, not generic)
  - Meaningful preferred/bonus qualifications

Exclude completely:
  - Company boilerplate ("We are a global leader in…")
  - EEO / diversity statements, legal text, agency notes
  - Benefits, perks, pay ranges
  - Generic filler ("strong communication skills", "team player")

Output: plain prose, no bullets, no headers. Under 200 words. Dense and specific.
"""

print("Chunk extraction prompt:")
print(CHUNK_PROMPT)

# ── Load pre-generated chunks ──────────────────────────────────────────────
if not CHUNKS_CSV.exists():
    print("⚠  jobs_chunks.csv not found.")
    print("   Run:  python scripts/generate_chunks.py")
    print("   Cost: ~$0.14 for 500 jobs  (run once, commit the file)")
else:
    chunks_df = pd.read_csv(CHUNKS_CSV)
    print(f"Loaded {len(chunks_df)} pre-chunked jobs from {CHUNKS_CSV.name}")
    print()

    # Compare raw vs chunk for one job
    raw_df    = pd.read_csv("../data/jobs.csv")
    sample_id = chunks_df.iloc[0]["id"]
    raw_row   = raw_df[raw_df["id"] == sample_id].iloc[0]
    chunk_row = chunks_df.iloc[0]

    raw_len   = len(str(raw_row["job_description"]))
    chunk_len = len(str(chunk_row["chunk_text"]))

    print(f"--- {chunk_row['job_title']} @ {chunk_row['company_name']} ---")
    print(f"Raw description : {raw_len} chars")
    print(f"LLM chunk       : {chunk_len} chars  ({chunk_len*100//raw_len}% of original)")
    print()
    print("CHUNK TEXT:")
    print(chunk_row["chunk_text"])

Chunk extraction prompt:

You are a technical recruiter assistant extracting a clean indexing chunk from a job description.

Extract and include:
  - Role title, seniority, and company domain
  - Required technical skills and tools (languages, frameworks, platforms)  
  - Years of experience required
  - 1-2 lines of key responsibilities (specific, not generic)
  - Meaningful preferred/bonus qualifications

Exclude completely:
  - Company boilerplate ("We are a global leader in…")
  - EEO / diversity statements, legal text, agency notes
  - Benefits, perks, pay ranges
  - Generic filler ("strong communication skills", "team player")

Output: plain prose, no bullets, no headers. Under 200 words. Dense and specific.

Loaded 500 pre-chunked jobs from jobs_chunks.csv

--- Manager, Software Development @ Fiserv ---
Raw description : 4937 chars
LLM chunk       : 710 chars  (14% of original)

CHUNK TEXT:
Manager, Software Development at Fiserv in the Fintech industry. Required technical skill

In [6]:
import chromadb
import time

CHROMA_PATH = "../chroma_db"   # ← same persistent store as the service
COLLECTION  = "jobs_v2"
DISTANCE_FN = "cosine"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

existing = [c.name for c in chroma_client.list_collections()]

if COLLECTION in existing:
    collection = chroma_client.get_collection(COLLECTION)
    print(f"Loaded existing index: {collection.count()} jobs  (no API calls)")

else:
    if not CHUNKS_CSV.exists():
        raise FileNotFoundError(
            "jobs_chunks.csv not found. Run scripts/generate_chunks.py first."
        )

    print(f"Building index  (distance_fn={DISTANCE_FN}) …")
    collection = chroma_client.create_collection(
        name=COLLECTION,
        metadata={"hnsw:space": DISTANCE_FN},
    )

    active = chunks_df[chunks_df["is_active"] == True].copy()
    print(f"Active jobs: {len(active)}")

    BATCH_SIZE = 50
    rows       = active.to_dict("records")
    inserted   = 0
    t0         = time.time()

    for i in range(0, len(rows), BATCH_SIZE):
        batch = rows[i : i + BATCH_SIZE]
        texts = [str(r["chunk_text"] or "") for r in batch]

        resp = client.embeddings.create(model="text-embedding-3-small", input=texts)
        embeddings = [e.embedding for e in resp.data]

        collection.add(
            ids        = [str(r["id"]) for r in batch],
            embeddings = embeddings,
            documents  = texts,
            metadatas  = [
                {
                    "job_title"  : str(r.get("job_title",    "") or ""),
                    "company"    : str(r.get("company_name", "") or ""),
                    "location"   : str(r.get("location",     "") or ""),
                    "salary"     : str(r.get("salary",       "") or ""),
                    "apply_link" : str(r.get("apply_link",   "") or ""),
                    "skills"     : str(r.get("skills",       "") or ""),
                    "is_active"  : 1,
                }
                for r in batch
            ],
        )
        inserted += len(batch)
        print(f"  Indexed {inserted}/{len(rows)} …", end="\r")

    elapsed = time.time() - t0
    print(f"\nDone: {inserted} jobs in {elapsed:.1f}s  →  {CHROMA_PATH}/")

Loaded existing index: 500 jobs  (no API calls)


---
## Step 4 — Two-Level Retrieval

Semantic search is powerful but broad — a query for "Java 8 yrs fintech" can return
a project-manager role at 90% similarity if we don't constrain the search.

**Level 1 — CSV metadata filter** (fast, O(1)):
- `is_active = True`
- `experience_required ≤ candidate's years`
- Optional: location, job_type
- Produces a set of **candidate job IDs**

**Level 2 — ChromaDB `where` + vector similarity** (semantic):
- Restricts vector search to the filtered ID set
- Ranks by cosine similarity to the candidate's skill query
- Returns top-N most relevant roles

```
CSV DataFrame filter  →  [id1, id2, id3 … id280]
      ↓
ChromaDB.query(where={id: $in [id1...id280]}, vector=query_embedding)
      ↓
Ranked: [Java SDE @ Fiserv 89%, Platform Eng @ NVIDIA 86%, …]
```

This is the standard production RAG pattern: **metadata pre-filter + semantic re-rank**.

In [7]:
def search_jobs_two_level(skills, min_years=0, domain=None, n_results=5):
    """
    Semantic job search using ChromaDB cosine similarity.
    Embeds a natural-language query from candidate's skills → finds closest job chunks.
    """
    domain_str  = f" in {domain}" if domain else ""
    query_text  = (
        f"Software engineer with {min_years}+ years{domain_str}. "
        f"Skills: {', '.join(skills)}."
    )
    query_vector = embed(query_text)
    print(f"Query: {query_text}")

    raw = collection.query(
        query_embeddings = [query_vector],
        n_results        = n_results,
        include          = ["metadatas", "distances", "documents"],
    )

    jobs = []
    for meta, dist, doc in zip(raw["metadatas"][0], raw["distances"][0], raw["documents"][0]):
        score = max(1, min(99, round((1 - dist) * 100)))
        jobs.append({
            "job_title"     : meta["job_title"],
            "company"       : meta["company"],
            "location"      : meta["location"],
            "semantic_score": score,
            "chunk_snippet" : doc[:180] + "…",
        })

    return jobs, {"query": query_text}

print("search_jobs_two_level() ready")

search_jobs_two_level() ready


In [8]:
ARJUN_SKILLS = ["Java", "Go", "Kafka", "Redis", "distributed systems"]

results, debug = search_jobs_two_level(
    skills    = ARJUN_SKILLS,
    min_years = 8,
    n_results = 5,
)

print(f"Level 2 (vector search) : {len(results)} results")
print(f"Query: {debug['query']}")
print()
print(f"{'Score':>7}  {'Company':<25}  Job Title")
print("-" * 75)
for r in results:
    print(f"  {r['semantic_score']:>4}%  {r['company']:<25}  {r['job_title'][:45]}")
print()
print("── Chunk snippet (what was actually embedded) ────────────────────")
print(results[0]["chunk_snippet"])

Query: Software engineer with 8+ years. Skills: Java, Go, Kafka, Redis, distributed systems.
Level 2 (vector search) : 5 results
Query: Software engineer with 8+ years. Skills: Java, Go, Kafka, Redis, distributed systems.

  Score  Company                    Job Title
---------------------------------------------------------------------------
    56%  Mastercard                 Director, Software Engineering
    55%  Mastercard                 Senior Software Engineer (Java Full stack)
    54%  NVIDIA                     System Software Engineer, Distributed Systems
    54%  Fiserv                     Manager, Software Development
    53%  NVIDIA                     Senior Full-Stack Software Engineer, Storage 

── Chunk snippet (what was actually embedded) ────────────────────
Director, Software Engineering at Mastercard in the payments industry. Required technical skills include Java/Spring Boot, Go, C++, microservices, REST APIs, event-driven architect…


---
## Step 5 — Side-by-Side Comparison

In [9]:
ARJUN_SKILLS = ["Java", "Go", "Kafka", "Redis", "distributed systems"]
ARJUN_YEARS  = 8

print("KEYWORD SEARCH (NB05 approach)")
print("─" * 50)
keyword_results = search_jobs_keyword(skills=ARJUN_SKILLS, min_years=ARJUN_YEARS)
if keyword_results:
    for r in keyword_results:
        print(f"  {r['match_score']}%  {r['job_title']} @ {r['company']}")
else:
    print("  (no results — skills column is empty on real data)")

print()
print("SEMANTIC SEARCH (NB06 — two-level retrieval)")
print("─" * 50)
semantic_results, _ = search_jobs_two_level(skills=ARJUN_SKILLS, min_years=ARJUN_YEARS, n_results=5)
for r in semantic_results:
    print(f"  {r['semantic_score']}%  {r['job_title'][:45]} @ {r['company']}")

print()
print("Same candidate. Same job database. Different engine.")


KEYWORD SEARCH (NB05 approach)
──────────────────────────────────────────────────
  (no results — skills column is empty on real data)

SEMANTIC SEARCH (NB06 — two-level retrieval)
──────────────────────────────────────────────────
Query: Software engineer with 8+ years. Skills: Java, Go, Kafka, Redis, distributed systems.
  56%  Director, Software Engineering @ Mastercard
  55%  Senior Software Engineer (Java Full stack) @ Mastercard
  54%  System Software Engineer, Distributed Systems @ NVIDIA
  54%  Manager, Software Development @ Fiserv
  53%  Senior Full-Stack Software Engineer, Storage  @ NVIDIA

Same candidate. Same job database. Different engine.


---
## Step 6 — Plug Back into the Agent

The agent from NB05 calls `search_jobs(skills, min_years)` — that's the tool contract.
We keep the same signature. Only the implementation changes.

**The agent doesn't know anything changed.**

In [10]:
# ── Drop-in replacement — same signature as NB05, two-level engine inside ──

def search_jobs(skills: list, domain: str = None, min_years: int = 0, n_results: int = 4) -> list:
    """Same tool schema as NB05. Two-level retrieval underneath."""
    results, _ = search_jobs_two_level(
        skills=skills, min_years=min_years, domain=domain, n_results=n_results
    )
    for r in results:
        r["match_score"] = r.pop("semantic_score")
    return results


print(f"ChromaDB index: {collection.count()} docs loaded")
print("search_jobs() wired to two-level retrieval.\n")

test = search_jobs(skills=["Java", "Kafka", "distributed systems"], min_years=8)
print(f"Quick test — {len(test)} results:")
for r in test:
    print(f"  {r['match_score']}%  {r['job_title']} @ {r['company']} ({r['location']})")

ChromaDB index: 500 docs loaded
search_jobs() wired to two-level retrieval.

Query: Software engineer with 8+ years. Skills: Java, Kafka, distributed systems.
Quick test — 4 results:
  58%  Senior Software Engineer (Java Full stack) @ Mastercard (Pune, India)
  57%  Manager, Software Development @ Fiserv (Pune - Trion Business Park, India)
  55%  Lead Software Engineer @ Mastercard (Pune, India)
  55%  Director, Software Engineering @ Mastercard (Pune, India)


In [11]:
import requests

# ── Tool implementations ───────────────────────────────────────────────────

def read_github(username: str) -> dict:
    """Fetch real GitHub profile via public API. Gracefully handles rate-limits."""
    demo_username = "mahadevTW"
    try:
        user  = requests.get(f"https://api.github.com/users/{demo_username}", timeout=5).json()
        repos = requests.get(f"https://api.github.com/users/{demo_username}/repos?sort=stars&per_page=10", timeout=5).json()
    except Exception as e:
        return {"error": str(e)}

    if not isinstance(user, dict) or "login" not in user:
        return {"error": user.get("message", "GitHub API unavailable") if isinstance(user, dict) else "GitHub API unavailable"}
    if not isinstance(repos, list):
        return {"error": repos.get("message", "Could not fetch repos") if isinstance(repos, dict) else "Could not fetch repos"}

    lang_count = {}
    for r in repos:
        if r.get("language"):
            lang_count[r["language"]] = lang_count.get(r["language"], 0) + 1
    return {
        "username"     : demo_username,
        "name"         : user.get("name"),
        "company"      : user.get("company"),
        "public_repos" : user.get("public_repos"),
        "followers"    : user.get("followers"),
        "top_languages": sorted(lang_count, key=lang_count.get, reverse=True)[:3],
        "top_repos"    : [{"name": r["name"], "stars": r["stargazers_count"],
                           "lang": r["language"], "desc": r["description"],
                           "last_pushed": r["pushed_at"][:10]} for r in repos[:5]],
    }

def ask_candidate(question: str) -> dict:
    """Ask the candidate a clarification question. Reads the answer from stdin."""
    print(f"\n  Counsellor : {question}")
    answer = input("  You        : ").strip() or "No answer."
    return {"question": question, "answer": answer}

def finish_session(recommendation: str, top_jobs: list, next_steps: list) -> dict:
    return {"status": "completed", "recommendation": recommendation,
            "top_jobs": top_jobs, "next_steps": next_steps}

def execute_tool(name: str, args: dict):
    if name == "ask_candidate" : return ask_candidate(**args)
    if name == "read_github"   : return read_github(**args)
    if name == "search_jobs"   : return search_jobs(**args)   # ← semantic (defined above)
    if name == "finish_session": return finish_session(**args)
    raise ValueError(f"Unknown tool: {name}")

# Tool schemas — identical to NB05
AGENT_TOOLS = [
    {"type":"function","function":{"name":"ask_candidate","description":"Ask the candidate one clarification question.","parameters":{"type":"object","properties":{"question":{"type":"string"}},"required":["question"]}}},
    {"type":"function","function":{"name":"read_github","description":"Fetch the candidate's GitHub profile.","parameters":{"type":"object","properties":{"username":{"type":"string"}},"required":["username"]}}},
    {"type":"function","function":{"name":"search_jobs","description":"Search open roles matching the candidate's skills and experience. Call ONLY after discovery and GitHub check.","parameters":{"type":"object","properties":{"skills":{"type":"array","items":{"type":"string"}},"domain":{"type":"string"},"min_years":{"type":"integer"}},"required":["skills","min_years"]}}},
    {"type":"function","function":{"name":"finish_session","description":"Call when ALL phases are done: questions answered, GitHub verified, jobs found.","parameters":{"type":"object","properties":{"recommendation":{"type":"string"},"top_jobs":{"type":"array","items":{"type":"string"},"description":"Copy the exact job_title and company from search_jobs results. Format each as: 'job_title at company (location)'. Never invent or generalise company names."},"next_steps":{"type":"array","items":{"type":"string"}}},"required":["recommendation","top_jobs","next_steps"]}}},
]

AGENT_GOAL = """
You are an AI career counsellor. Work through FOUR phases in strict order.

PHASE 1 — DISCOVERY (ask_candidate): Fill missing fields — salary, notice period,
reason for leaving, relocation, gap explanations. One question at a time.

PHASE 2 — VERIFICATION (read_github): Check GitHub if username is available.

PHASE 3 — JOB SEARCH (search_jobs): Only after phases 1 and 2.

PHASE 4 — FINISH (finish_session): Grounded recommendation referencing real data.
In top_jobs, use the EXACT job_title and company from search_jobs — never make up names.
"""

ARJUN_PARTIAL = {
    "name"           : "Arjun Mehta",
    "current_role"   : "Senior SDE-II at Flipkart",
    "total_experience": 8,
    "top_skills"     : ["Java", "Go", "Kafka", "Redis", "distributed systems"],
    "career_goal"    : "Senior IC or Tech Lead in fintech/infra",
    "github"         : "arjun91",
    "career_gaps"    : [
        {"from": "Aug 2017", "to": "Mar 2018", "duration_months": 7,  "reason_given": False},
        {"from": "Feb 2021", "to": "Sep 2021", "duration_months": 7,  "reason_given": False},
    ],
    "salary_expectation" : None,
    "notice_period"      : None,
    "reason_for_change"  : None,
    "open_to_relocation" : None,
}

print("Agent wired with semantic search_jobs. Ready to run.")

Agent wired with semantic search_jobs. Ready to run.


In [12]:
PHASE_LABELS = {
    "ask_candidate" : "PHASE 1 — Discovery",
    "read_github"   : "PHASE 2 — Verification",
    "search_jobs"   : "PHASE 3 — Job Search",
    "finish_session": "PHASE 4 — Finish",
}

def run_agent(profile: dict) -> dict:
    messages = [
        {"role": "system", "content": AGENT_GOAL},
        {"role": "user",   "content": f"Candidate profile:\n{json.dumps(profile, indent=2)}\n\nBegin the assessment."},
    ]
    current_phase = None

    for turn in range(1, 25):
        resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=AGENT_TOOLS)
        msg  = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"Turn {turn}: Agent stopped without a tool call.")
            return {}

        for tc in msg.tool_calls:
            args   = json.loads(tc.function.arguments)
            result = execute_tool(tc.function.name, args)

            label = PHASE_LABELS.get(tc.function.name)
            if label != current_phase:
                current_phase = label
                print(f"\n{'─'*55}")
                print(f"  {label}")
                print(f"{'─'*55}")

            if tc.function.name == "ask_candidate":
                print(f"  Q: {args['question']}")
                print(f"  A: {result['answer']}")
            elif tc.function.name == "read_github":
                if "error" in result:
                    print(f"  GitHub: unavailable ({result['error']}) — skipping")
                else:
                    print(f"  {result.get('name')} | {result.get('public_repos')} repos | {result.get('top_languages')}")
            elif tc.function.name == "search_jobs":
                if result:
                    print(f"  Found {len(result)} jobs — top: {result[0]['job_title']} @ {result[0]['company']} ({result[0]['match_score']}%)")
                else:
                    print("  Found 0 jobs")
            elif tc.function.name == "finish_session":
                print(f"\n  Done in {turn} turns.")
                return result

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    return {}


print("=" * 55)
print("AGENT SESSION — semantic search_jobs")
print("=" * 55)

output = run_agent(ARJUN_PARTIAL)

print()
print("=" * 55)
print("FINAL OUTPUT")
print("=" * 55)
print(f"\nAssessment:")
print(f"  {output.get('recommendation')}")
print(f"\nTop Jobs:")
for job in output.get("top_jobs", []):
    print(f"  • {job}")
print(f"\nNext Steps:")
for step in output.get("next_steps", []):
    print(f"  → {step}")


AGENT SESSION — semantic search_jobs

  Counsellor : What is your salary expectation for your next role?

───────────────────────────────────────────────────────
  PHASE 1 — Discovery
───────────────────────────────────────────────────────
  Q: What is your salary expectation for your next role?
  A: 30LPA

  Counsellor : What is your notice period at Flipkart?
  Q: What is your notice period at Flipkart?
  A: 2 months

  Counsellor : What is your reason for leaving your current position at Flipkart?
  Q: What is your reason for leaving your current position at Flipkart?
  A: better role

  Counsellor : Are you open to relocation? If yes, which locations are you considering?
  Q: Are you open to relocation? If yes, which locations are you considering?
  A: Yes

───────────────────────────────────────────────────────
  PHASE 2 — Verification
───────────────────────────────────────────────────────
  GitHub: unavailable (API rate limit exceeded for 3.108.156.89. (But here's the good news:

---
## The Aha Moment

What changed between NB05 and NB06?

| | NB05 | NB06 |
|---|---|---|
| Agent goal | unchanged | unchanged |
| Tool schemas | unchanged | unchanged |
| Agent loop | unchanged | unchanged |
| LLM model | unchanged | unchanged |
| `search_jobs` implementation | keyword overlap | cosine similarity |
| Jobs accessible | 0 (empty skills column) | 499 real jobs |

**One function. The whole system got smarter.**

---

This is the pattern you'll use everywhere:

```
Text  →  Embedding  →  Vector Store  →  Nearest Neighbours  →  LLM
```

It has a name: **Retrieval-Augmented Generation (RAG)**.
You've just built it from scratch.

---

```
Intelligence is not in the LLM.

It's in the tools. The retrieval. The index.
The system you design around the LLM.
```

> **Understand → Infer → Act → Agent → *Semantic Retrieval***